In [79]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


## Load The Data
The below cell loads in all of the provided data into two Data Frames.
titanic_train_data - Provides the data used to train the model with before submission
titanic_validation_data - Provides the data used to predict the survival of each individual passenger aboard the Titanic

In [80]:
titanic_train_file_path = '/kaggle/input/competitions/titanic/train.csv'
titanic_train_data = pd.read_csv(titanic_train_file_path)
titanic_validation_file_path = '/kaggle/input/competitions/titanic/test.csv'
titanic_validation_data = pd.read_csv(titanic_validation_file_path)

## Create The Variables
We start by importing the train_test_split function from the model_selection library within sklearn

Next, we decide which columns we want the model to take into consideration for its prediction

Then, we define our variables X and y:
X - Contains all of the data within the columns defined in test_columns
y - Contains the survival column, which acts as our answer key

Finally, we define four variables to use while testing the model's various arguments, each defined using the train_test_split function we imported earlier. This function splits the data into a train section, which we use to train the model, and a test section, which we feed the model to give us a prediction, and compare it with the answers to get our accuracy

In [81]:
from sklearn.model_selection import train_test_split

test_columns = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Age', 'Fare']
X = pd.get_dummies(titanic_train_data[test_columns])
y = titanic_train_data.Survived

model_train_X, test_X, model_train_y, test_y = train_test_split(X,y,random_state=1)

## Creating Our Model
For this project, I've chosen to use a Random Forest model. We use the classifier because our predictions are categorical rather than continuous.

We start by importing the RandomForestClassifier function from the ensemble library, as well as the mean_absolute_error function from the metrics library, both within sklearn. We will use these functions to create our model and test its accuracy.

For now, any model we define will be for the sole purpose of testing its various arguments. Before creating our final model, we want to test various inputs for each argument the model has in order to optimize our model for our data set.

## Function One: How Many Trees In The Forest
This first function takes various n_estimators values (which dictate the number of trees within the random forest) alongside our train_test_split variables and returns the mean absolute error of our model, which we turn into the percentage of correct predictions. We then use a for loop to iterate through various tree values, before plugging into the function to get each tree value's accuracy score. We use an if statement to figure out which one is the best (aka has the highest accuracy score), and then print it out for us to see.

In [82]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error

def n_estimators_accuracy(n_estimators,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(n_estimators=n_estimators,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy_trees = 0
best_n_estimators = 0
for n_estimators in [10,50,100,150,200,300,500,1000]:
    accuracy = n_estimators_accuracy(n_estimators,model_train_X,test_X,model_train_y,test_y)
    if accuracy > highest_accuracy_trees:
        highest_accuracy_trees = accuracy
        best_n_estimators = n_estimators
print('Best N-Estimators:', best_n_estimators, '\nAccuracy:', highest_accuracy_trees)

Best N-Estimators: 150 
Accuracy: 79.82062780269058


## Function Two: Quality Of Split
This second function allows us to figure out which criterion is the most accurate for our function. We only need to test for gini and entropy because log_loss is the same as entropy in the context of our model. Through using this function, we discover that using 'gini' provides us with a higher accuracy score. The functions within this notebook will all look very similar, as their purposes are all to calculate the most effective argument for every argument of a Random Forest Classifier.

In [83]:
def criterion_accuracy(criterion,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(criterion=criterion,n_estimators=150,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy_criterion = 0
best_criterion = ''
for criterion in ['gini','entropy']:
    accuracy = criterion_accuracy(criterion,model_train_X,test_X,model_train_y,test_y)
    if accuracy >= highest_accuracy_criterion:
        highest_accuracy_criterion = accuracy
        best_criterion = criterion
print('Best Criterion:', best_criterion, '\nAccuracy:', highest_accuracy_criterion)

Best Criterion: gini 
Accuracy: 79.82062780269058


## Function Three: Max Depth
This third function allows us to optimize for the maximum depth of each individual tree that would allow us to have the most accurate predictions. We originally used a spread of values [2,3,5,8,12,20], but after our first test, we got a result of 3, so we added 4 to ensure that 3 was the best among those tested on this validation split.

In [84]:
def max_depth_accuracy(max_depth,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(max_depth=max_depth,criterion='gini',n_estimators=150,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy_max_depth = 0
best_max_depth = ''
for max_depth in [2,3,4,5,8,12,20,None]:
    accuracy = max_depth_accuracy(max_depth,model_train_X,test_X,model_train_y,test_y)
    if accuracy > highest_accuracy_max_depth:
        highest_accuracy_max_depth = accuracy
        best_max_depth = max_depth
print('Best Max Depth:', best_max_depth, '\nAccuracy:', highest_accuracy_max_depth)

Best Max Depth: 3 
Accuracy: 80.26905829596413


## Function Four: Minimum Leaf Samples
This fourth function helps us optimize for the smallest amount of samples required to form a new leaf within each tree in the forest. We use a spread of values [1,2,4,8,16,32] because it helps us get better perspective on if more samples or less samples increases accuracy. For this specific argument, the most accurate value was 1, so we didn't need to add more values to the spread to narrow it down.

In [85]:
def min_leaf_samples_accuracy(min_samples_leaf,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(min_samples_leaf=min_samples_leaf,max_depth=3,criterion='gini',n_estimators=150,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy_min_samples_leaf = 0
best_min_samples_leaf = ''
for min_samples_leaf in [1,2,4,8,16,32]:
    accuracy = min_leaf_samples_accuracy(min_samples_leaf,model_train_X,test_X,model_train_y,test_y)
    if accuracy > highest_accuracy_min_samples_leaf:
        highest_accuracy_min_samples_leaf = accuracy
        best_min_samples_leaf = min_samples_leaf
print('Best Min Samples Leaf:', best_min_samples_leaf, '\nAccuracy:', highest_accuracy_min_samples_leaf)

Best Min Samples Leaf: 1 
Accuracy: 80.26905829596413


In [86]:
def max_features_accuracy(max_features,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(max_features=max_features,min_samples_leaf=1,max_depth=3,criterion='gini',n_estimators=150,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy_max_features = 0
best_max_features = ''
for max_features in range(1, model_train_X.shape[1] + 1):
    accuracy = max_features_accuracy(max_features,model_train_X,test_X,model_train_y,test_y)
    if accuracy > highest_accuracy_max_features:
        highest_accuracy_max_features = accuracy
        best_max_features = max_features
print('Best Max Features:', best_max_features, '\nAccuracy:', highest_accuracy_max_features)

Best Max Features: 2 
Accuracy: 80.26905829596413


## Function Six: Testing Two Arguments Simultaneously
Now that we have tested the five main arguments that would have the biggest impact on our accuracy score, we can look back to see which two would work the best together. The main drawback to testing one argument at a time is that we can't see how their varying values would work together. Through reflection, I realize that setting max_depth = 3 before testing min_samples_leaf limits the tests effectiveness, as we are already set to a very shallow tree. This is why we will implement one more function to test these two arguments simultaneously, to ensure that we have the correct values for our tree.
Even though this test ended up being redundant, it's always important to double check your process for flaws and address them to ensure your model is as accurate as can be.

In [87]:
def two_argument_accuracy(max_depth,min_samples_leaf,model_train_X,test_X,model_train_y,test_y):
    model = RandomForestClassifier(max_depth=max_depth,min_samples_leaf=min_samples_leaf,max_features=2,criterion='gini',n_estimators=150,random_state=1)
    model.fit(model_train_X,model_train_y)
    prediction = model.predict(test_X)
    mae = mean_absolute_error(test_y,prediction)
    accuracy = (1 - mae) * 100
    return accuracy

highest_accuracy = 0
best_max_depth_2 = 0
best_min_samples_leaf_2 = 0
for max_depth in [3,5,8,None]:
    for min_samples_leaf in [1,4,8]:
        accuracy = two_argument_accuracy(max_depth,min_samples_leaf,model_train_X,test_X,model_train_y,test_y)
        if accuracy > highest_accuracy:
            highest_accuracy = accuracy
            best_max_depth_2 = max_depth
            best_min_samples_leaf_2 = min_samples_leaf
            
print('Best Max Depth:', best_max_depth_2, '\nBest Min Samples Leaf', best_min_samples_leaf_2, '\nAccuracy:', highest_accuracy)

Best Max Depth: 3 
Best Min Samples Leaf 1 
Accuracy: 80.26905829596413


## Changing Test Columns
While an accuracy score of 80.27% from a train/test split is the best result among the configurations tested so far. It's important to note that there are other factors that can affect the model's accuracy outside of the arguments. One of the is editing the entries within test_columns. After creating all of the functions, I reflected on my code and realized that I had intuitively selected the columns that I was using, so I tried adding the column 'Embarked'. After updating all of my code, these were the changes.

1. Trees: 150 -> 300
2. Criterion: 'Gini' -> 'Gini'
3. Max Depth: 3 -> 3
4. Min Leaf: 1 -> 1
5. Max Features: 2 -> 5

Since there was no change in the accuracy (80.27% = 80.27%), I decided to stick with my original list without the 'Embarked' column, as 150 trees runs faster than 300 trees, and I want the most efficient model.

## Further Testing
After more speculation, I realized that the only new column to add wasn't 'Embarked', but also 'Name'. However, not just the names themselves, but the titles too. A passenger with a more important title could be more likely to survive the titanic compared to a lower-class passenger. To do this, I had to rebrand the 'Name' column into the 'Title' column, and strip everything away except for their titles and use one-hot encoding so that the model could understand the data. This can be seen within the code cell below.
After re-running all of my functions with the new 'Title' column, here are my new arguments for my Random Forest Classifier:

1. n_estimators: 150 -> 10
2. criterion: 'gini' -> 'entropy'
3. max_depth: 3 -> 12
4. min_samples_leaf: 1 -> 1
5. max_features: 2 -> 4

After updating the model, the new accuracy score was 81.61%, which is a 1.35 percentage-point improvement on this validation split, which only provided an 80.27% accuracy.

In [88]:
titanic_train_data["Title"] = (
    titanic_train_data["Name"]
    .str.split(",").str[1]
    .str.split(".").str[0]
    .str.strip()
)
titanic_validation_data["Title"] = (
    titanic_validation_data["Name"]
    .str.split(",").str[1]
    .str.split(".").str[0]
    .str.strip()
)
test_columns_updated = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Age', 'Fare', 'Title']

In [89]:
temp_X = pd.get_dummies(titanic_train_data[test_columns_updated])
temp_y = titanic_train_data.Survived

model_train_temp_X, test_temp_X, model_train_temp_y, test_temp_y = train_test_split(temp_X,temp_y,random_state=1)

temp_model = RandomForestClassifier(
    n_estimators = 10,
    criterion = 'entropy',
    max_depth = 12,
    min_samples_leaf = 1,
    max_features = 4,
    random_state = 1
)

temp_model.fit(model_train_temp_X, model_train_temp_y)
temp_model_predictions = temp_model.predict(test_temp_X)

temp_model_accuracy = ((1 - mean_absolute_error(test_temp_y, temp_model_predictions)) * 100)
print("Accuracy with titles column:", temp_model_accuracy)

Accuracy with titles column: 81.61434977578476


In [90]:
train_y = titanic_train_data.Survived
train_X = pd.get_dummies(titanic_train_data[test_columns_updated])

val_X = pd.get_dummies(titanic_validation_data[test_columns_updated])
val_X = val_X.reindex(columns=train_X.columns, fill_value=0)

In [91]:
from sklearn.ensemble import RandomForestClassifier

titanic_forest_model = RandomForestClassifier(max_features=4,min_samples_leaf=1,max_depth=12,criterion='entropy',n_estimators=10,random_state=1)
titanic_forest_model.fit(train_X, train_y)

survival_predictions = titanic_forest_model.predict(val_X)

In [92]:
output = pd.DataFrame({'PassengerId': titanic_validation_data.PassengerId,
                      'Survived': survival_predictions})
output.to_csv('submission.csv',index=False)

## Final Submission
After implementing all of the changes recommended by the results of my functions, my final accuracy score is 79.186%.